In [6]:
# %% [markdown]
# # Оценка качества регрессии: вычисление метрик
# 
# Данный ноутбук загружает CSV-файл с колонками `test` и `pred`,
# вычисляет основные метрики регрессии и выводит их в виде таблицы.
# 
# **Метрики**: MAE, MAPE, WAPE, MSE, RMSE, MSLE, RMSLE.

# %%
import pandas as pd
import numpy as np

# %% [markdown]
# ## 1. Загрузка данных

# %%

file_path = 'test_pred.csv'

df = pd.read_csv(file_path)




required_columns = ['test', 'pred']
if not all(col in df.columns for col in required_columns):
    raise ValueError(f"Файл должен содержать колонки: {required_columns}")


df = df.dropna(subset=required_columns)


y_true = df['test'].values
y_pred = df['pred'].values

print(f"Загружено {len(y_true)} наблюдений.\n")

# Удаляем строки с отрицательными предсказаниями
initial_len = len(df)
df = df[df['pred'] >= 0]
print(f"Удалено {initial_len - len(df)} строк (отрицательные pred)")


df = df[df['test'] >= 0]

# Обновляем массивы
y_true = df['test'].values
y_pred = df['pred'].values

print(f"Осталось наблюдений: {len(y_true)}")

# %% [markdown]
# ## 2. Функции для вычисления метрик

# %%
def mae(y_true, y_pred):
    """Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error (в процентах)"""
    # Защита от деления на ноль: где y_true == 0, вклад не учитываем
    mask = y_true != 0
    if np.sum(mask) == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def wape(y_true, y_pred):
    """Weighted Absolute Percentage Error (в долях, не в процентах)"""
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def mse(y_true, y_pred):
    """Mean Squared Error"""
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(mse(y_true, y_pred))

def msle(y_true, y_pred):
    """Mean Squared Logarithmic Error (с добавлением 1 для устойчивости)"""
    log_true = np.log1p(y_true)   # ln(1 + y_true)
    log_pred = np.log1p(y_pred)   # ln(1 + y_pred)
    return np.mean((log_true - log_pred) ** 2)

def rmsle(y_true, y_pred):
    """Root Mean Squared Logarithmic Error"""
    return np.sqrt(msle(y_true, y_pred))

# %% [markdown]
# ## 3. Вычисление метрик

# %%
metrics = {
    'MAE': mae(y_true, y_pred),
    'MAPE (%)': mape(y_true, y_pred),
    'WAPE': wape(y_true, y_pred),
    'MSE': mse(y_true, y_pred),
    'RMSE': rmse(y_true, y_pred),
    'MSLE': msle(y_true, y_pred),
    'RMSLE': rmsle(y_true, y_pred)
}

# %% [markdown]
# ## 4. Вывод результатов в виде таблицы

# %%
results_df = pd.DataFrame(list(metrics.items()), columns=['Метрика', 'Значение'])

results_df['Значение'] = results_df['Значение'].map(lambda x: f"{x:.4f}")

print("Результаты оценки качества модели:\n")
print(results_df.to_string(index=False))



Загружено 24788 наблюдений.

Удалено 2 строк (отрицательные pred)
Осталось наблюдений: 24786
Результаты оценки качества модели:

 Метрика Значение
     MAE   3.7866
MAPE (%)  23.5379
    WAPE   0.1767
     MSE  40.1626
    RMSE   6.3374
    MSLE   0.0835
   RMSLE   0.2889
